In [1]:
import sys

print("Python version:", sys.version)

Python version: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]


In [2]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch version: 2.11.0+cpu
CUDA available: False


In [3]:
!pip install -q safetensors

In [5]:
import safetensors

print("safetensors installed successfully")

safetensors installed successfully


In [6]:
import torch
import torch.nn as nn

class LabModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(5000, 5000),
            nn.ReLU(),
            nn.Linear(5000, 5000),
            nn.ReLU(),
            nn.Linear(5000, 5000),
        )

    def forward(self, x):
        return self.network(x)


model = LabModel()

print(model)

LabModel(
  (network): Sequential(
    (0): Linear(in_features=5000, out_features=5000, bias=True)
    (1): ReLU()
    (2): Linear(in_features=5000, out_features=5000, bias=True)
    (3): ReLU()
    (4): Linear(in_features=5000, out_features=5000, bias=True)
  )
)


In [7]:
num_params = sum(p.numel() for p in model.parameters())

bytes_per_param = 4  # float32

model_size_bytes = num_params * bytes_per_param
model_size_mb = model_size_bytes / (1024 ** 2)

print("Number of parameters:", num_params)
print("Estimated weight size:", model_size_mb, "MB")

Number of parameters: 75015000
Estimated weight size: 286.1595153808594 MB


In [8]:
state = model.state_dict()

print(type(state))
print()

for name, tensor in state.items():
    print(
        name,
        "shape =", tuple(tensor.shape),
        "dtype =", tensor.dtype,
        "numel =", tensor.numel()
    )

<class 'collections.OrderedDict'>

network.0.weight shape = (5000, 5000) dtype = torch.float32 numel = 25000000
network.0.bias shape = (5000,) dtype = torch.float32 numel = 5000
network.2.weight shape = (5000, 5000) dtype = torch.float32 numel = 25000000
network.2.bias shape = (5000,) dtype = torch.float32 numel = 5000
network.4.weight shape = (5000, 5000) dtype = torch.float32 numel = 25000000
network.4.bias shape = (5000,) dtype = torch.float32 numel = 5000


Prediction — File sizes

The model contains 75,015,000 float32 parameters. Since each float32
parameter requires 4 bytes, I expect the raw tensor data to require
approximately 300,060,000 bytes, or about 286.16 MiB.

I expect the .pt and .safetensors files to be close to this size because
both contain the same tensor values. The .pt file may be slightly larger
because it also contains PyTorch serialization information.

I expect safetensors to store the tensor data in a simple packed format,
while .pt also stores information required to reconstruct the Python
object/state dictionary.

In [9]:
import torch

PT_PATH = "/content/lab_model.pt"

torch.save(model.state_dict(), PT_PATH)

print("Saved:", PT_PATH)

Saved: /content/lab_model.pt


In [10]:
from safetensors.torch import save_file

SAFE_PATH = "/content/lab_model.safetensors"

save_file(model.state_dict(), SAFE_PATH)

print("Saved:", SAFE_PATH)

Saved: /content/lab_model.safetensors


In [11]:
import os

pt_size = os.path.getsize(PT_PATH)
safe_size = os.path.getsize(SAFE_PATH)

print("PT size:")
print(f"  {pt_size:,} bytes")
print(f"  {pt_size / (1024**2):.2f} MiB")

print()

print("Safetensors size:")
print(f"  {safe_size:,} bytes")
print(f"  {safe_size / (1024**2):.2f} MiB")

PT size:
  300,063,085 bytes
  286.16 MiB

Safetensors size:
  300,060,528 bytes
  286.16 MiB


In [12]:
!ls -lh /content/lab_model.pt /content/lab_model.safetensors

-rw-r--r-- 1 root root 287M Sep 19 15:06 /content/lab_model.pt
-rw------- 1 root root 287M Sep 19 15:07 /content/lab_model.safetensors


In [13]:
import os

pt_size = os.path.getsize("/content/lab_model.pt")
safe_size = os.path.getsize("/content/lab_model.safetensors")

print("PT size:")
print(f"{pt_size:,} bytes")
print(f"{pt_size / (1024**2):.2f} MiB")

print("\nSafetensors size:")
print(f"{safe_size:,} bytes")
print(f"{safe_size / (1024**2):.2f} MiB")

PT size:
300,063,085 bytes
286.16 MiB

Safetensors size:
300,060,528 bytes
286.16 MiB


Prediction — Cold vs Warm Load

I predict that the cold-cache load will take longer than the warm-cache
load. After the first access, the operating system may keep the file's
pages in the page cache, allowing the second access to be served more
quickly.

I expect the difference to depend on the Colab machine and whether the
file was already cached. Therefore, I will use the measured values rather
than assuming a particular time.

I also predict that loading safetensors using mmap will have a much
smaller immediate RSS increase than loading the .pt file because mmap
maps the file into virtual memory instead of immediately materializing
the complete tensor data in user-space memory.

In [14]:
import os

def get_rss():
    with open("/proc/self/status", "r") as f:
        for line in f:
            if line.startswith("VmRSS:"):
                return int(line.split()[1]) * 1024

print("Current RSS:", get_rss(), "bytes")

Current RSS: 616144896 bytes


In [15]:
import os
import time
import gc

def get_rchar():
    with open("/proc/self/io", "r") as f:
        for line in f:
            if line.startswith("rchar:"):
                return int(line.split()[1])
    return None


def measure_load(load_function):
    gc.collect()

    rchar_before = get_rchar()

    start = time.perf_counter()

    obj = load_function()

    end = time.perf_counter()

    rchar_after = get_rchar()

    return {
        "object": obj,
        "time": end - start,
        "rchar": rchar_after - rchar_before
    }

In [16]:
result_pt = measure_load(
    lambda: torch.load(
        PT_PATH,
        weights_only=True
    )
)

print("PT load time:", result_pt["time"], "seconds")
print("PT rchar:", result_pt["rchar"], "bytes")

PT load time: 0.843909265000093 seconds
PT rchar: 300081617 bytes


In [17]:
import gc

gc.collect()

rss_before = get_rss()

start = time.perf_counter()

loaded_pt = torch.load(
    PT_PATH,
    weights_only=True
)

end = time.perf_counter()

rss_after = get_rss()

print("PT load time:", end - start, "seconds")
print("RSS before:", rss_before, "bytes")
print("RSS after:", rss_after, "bytes")
print("RSS delta:", rss_after - rss_before, "bytes")

PT load time: 0.6416273039999396 seconds
RSS before: 916496384 bytes
RSS after: 1216507904 bytes
RSS delta: 300011520 bytes


In [18]:
import gc

del loaded_pt
gc.collect()

print("Cleaned up previous PT object")

Cleaned up previous PT object


In [19]:
from safetensors.torch import load_file

result_safe = measure_load(
    lambda: load_file(SAFE_PATH)
)

print("Safetensors load time:", result_safe["time"], "seconds")
print("Safetensors rchar:", result_safe["rchar"], "bytes")

Safetensors load time: 0.07212324499960232 seconds
Safetensors rchar: 131 bytes


In [20]:
import gc
import time

del result_safe
gc.collect()

rss_before = get_rss()

start = time.perf_counter()

loaded_safe = load_file(SAFE_PATH)

end = time.perf_counter()

rss_after = get_rss()

print("Safetensors load time:", end - start, "seconds")
print("RSS before:", rss_before, "bytes")
print("RSS after:", rss_after, "bytes")
print("RSS delta:", rss_after - rss_before, "bytes")

Safetensors load time: 0.001422399000148289 seconds
RSS before: 918474752 bytes
RSS after: 918671360 bytes
RSS delta: 196608 bytes


In [21]:
del loaded_safe
gc.collect()

0

In [22]:
result_pt_warm = measure_load(
    lambda: torch.load(
        PT_PATH,
        weights_only=True
    )
)

print("Warm PT load time:", result_pt_warm["time"], "seconds")
print("Warm PT rchar:", result_pt_warm["rchar"], "bytes")

Warm PT load time: 0.9713559379997605 seconds
Warm PT rchar: 300081610 bytes


In [23]:
del result_pt_warm
gc.collect()

result_safe_warm = measure_load(
    lambda: load_file(SAFE_PATH)
)

print("Warm safetensors load time:", result_safe_warm["time"], "seconds")
print("Warm safetensors rchar:", result_safe_warm["rchar"], "bytes")

Warm safetensors load time: 0.007000712000262865 seconds
Warm safetensors rchar: 131 bytes


In [24]:
result_safe = measure_load(
    lambda: load_file(SAFE_PATH)
)

In [25]:
loaded_safe = load_file(SAFE_PATH)

Part 2 Prediction — Stride Tax

I predict that transposing a tensor will not copy its underlying data.
The original tensor and its transpose should therefore have the same
data pointer, while their stride tuples will be different.

I predict that calling .contiguous() on the transpose will allocate
new storage and physically copy the tensor values. Therefore the
time required for .contiguous() should increase as the number of
bytes being copied increases.

I also predict that safetensors will reject the non-contiguous
transpose because safetensors requires tensor data to be stored in
a contiguous canonical layout.

In [26]:
x = torch.randn(5000, 5000)

print("Shape:", x.shape)
print("Dtype:", x.dtype)
print("Contiguous:", x.is_contiguous())
print("Data pointer:", x.data_ptr())
print("Strides:", x.stride())

Shape: torch.Size([5000, 5000])
Dtype: torch.float32
Contiguous: True
Data pointer: 132168768442432
Strides: (5000, 1)


In [27]:
x_t = x.T

print("Shape:", x_t.shape)
print("Dtype:", x_t.dtype)
print("Contiguous:", x_t.is_contiguous())
print("Data pointer:", x_t.data_ptr())
print("Strides:", x_t.stride())

print()
print("Same data pointer:", x.data_ptr() == x_t.data_ptr())

Shape: torch.Size([5000, 5000])
Dtype: torch.float32
Contiguous: False
Data pointer: 132168768442432
Strides: (1, 5000)

Same data pointer: True


In [28]:
import time
import gc

gc.collect()

start = time.perf_counter()

x_contig = x_t.contiguous()

end = time.perf_counter()

print("Time for .contiguous():", end - start, "seconds")
print("Original pointer:", x.data_ptr())
print("Transpose pointer:", x_t.data_ptr())
print("Contiguous pointer:", x_contig.data_ptr())
print("x_t contiguous:", x_t.is_contiguous())
print("x_contig contiguous:", x_contig.is_contiguous())
print("x_t and x_contig share storage:",
      x_t.data_ptr() == x_contig.data_ptr())

Time for .contiguous(): 0.45169187400006194 seconds
Original pointer: 132168768442432
Transpose pointer: 132168768442432
Contiguous pointer: 132168668438592
x_t contiguous: False
x_contig contiguous: True
x_t and x_contig share storage: False


Prediction — .contiguous() scaling

I predict that larger tensors will require more time for .contiguous()
because more bytes have to be physically copied into new contiguous
storage.

Therefore, as tensor size increases, the .contiguous() time should
generally increase. The relationship may not be perfectly linear
because of memory allocation and system noise.

In [29]:
import time
import gc

sizes = [1000, 2000, 3000, 4000, 5000]

contiguous_times = []

for n in sizes:
    x = torch.randn(n, n)
    x_t = x.T

    gc.collect()

    start = time.perf_counter()
    x_contig = x_t.contiguous()
    end = time.perf_counter()

    elapsed = end - start
    bytes_copied = x.numel() * x.element_size()

    contiguous_times.append(elapsed)

    print(
        f"Size: {n}x{n} | "
        f"Bytes: {bytes_copied:,} | "
        f"Time: {elapsed:.6f} s"
    )

    del x, x_t, x_contig
    gc.collect()

Size: 1000x1000 | Bytes: 4,000,000 | Time: 0.013003 s
Size: 2000x2000 | Bytes: 16,000,000 | Time: 0.017867 s
Size: 3000x3000 | Bytes: 36,000,000 | Time: 0.041536 s
Size: 4000x4000 | Bytes: 64,000,000 | Time: 0.080264 s
Size: 5000x5000 | Bytes: 100,000,000 | Time: 0.128378 s


In [30]:
x = torch.randn(2000, 2000)
x_t = x.T

In [31]:
print("x contiguous:", x.is_contiguous())
print("x_t contiguous:", x_t.is_contiguous())

x contiguous: True
x_t contiguous: False


In [32]:
from safetensors.torch import save_file

try:
    save_file({"x": x_t}, "/content/transpose_test.safetensors")
    print("Unexpected: save succeeded")
except Exception as e:
    print("Expected error:")
    print(type(e).__name__)
    print(e)

Expected error:
ValueError
You are trying to save a non contiguous tensor: `x` which is not allowed. It either means you are trying to save tensors which are reference of each other in which case it's recommended to save only the full tensors, and reslice at load time, or simply call `.contiguous()` on your tensor to pack it before saving.


In [33]:
x_contig = x_t.contiguous()

print("x_contig contiguous:", x_contig.is_contiguous())

save_file(
    {"x": x_contig},
    "/content/transpose_fixed.safetensors"
)

print("Saved successfully!")

x_contig contiguous: True
Saved successfully!


In [34]:
from pathlib import Path
import os
import time
import gc

WORK = Path("/content/lab11_12_work")
WORK.mkdir(exist_ok=True)

RESULTS = {
    "formats": {},
    "stride": {},
    "trust": {},
    "gguf_real": {}
}

IS_LINUX = __import__("sys").platform.startswith("linux")


def rss_mb():
    if IS_LINUX:
        with open("/proc/self/status") as f:
            for line in f:
                if line.startswith("VmRSS:"):
                    return int(line.split()[1]) / 1024

    import resource
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024


def io_field(name):
    if not IS_LINUX:
        return None

    with open("/proc/self/io") as f:
        for line in f:
            if line.startswith(name):
                return int(line.split()[1])

    return None


def evict(path):
    if not IS_LINUX:
        return

    fd = os.open(path, os.O_RDONLY)

    try:
        os.posix_fadvise(
            fd,
            0,
            0,
            os.POSIX_FADV_DONTNEED
        )
    finally:
        os.close(fd)

In [35]:
def measure_pt(sd, path):

    torch.save(sd, path)

    file_mb = path.stat().st_size / 1e6

    # Cold load
    evict(path)

    start = time.perf_counter()
    rb0 = io_field("read_bytes")

    torch.load(path, weights_only=True)

    cold = time.perf_counter() - start

    rb1 = io_field("read_bytes")

    cold_read = (
        (rb1 - rb0) / 1e6
        if rb0 is not None and rb1 is not None
        else None
    )

    # Warm load
    start = time.perf_counter()

    torch.load(path, weights_only=True)

    warm = time.perf_counter() - start

    # RSS + rchar
    gc.collect()

    rss_before = rss_mb()
    rchar_before = io_field("rchar")

    obj = torch.load(path, weights_only=True)

    rss_after = rss_mb()
    rchar_after = io_field("rchar")

    rss_delta = rss_after - rss_before

    rchar_delta = (
        (rchar_after - rchar_before) / 1e6
        if rchar_before is not None and rchar_after is not None
        else None
    )

    del obj
    gc.collect()

    return {
        "file_mb": round(file_mb, 1),
        "cold_s": round(cold, 3),
        "warm_s": round(warm, 3),
        "cold_read_bytes_mb": round(cold_read, 1)
            if cold_read is not None else None,
        "load_rss_delta_mb": round(rss_delta, 1),
        "load_rchar_mb": round(rchar_delta, 1)
            if rchar_delta is not None else None
    }


def measure_safetensors(sd, path):

    save_file(sd, str(path))

    file_mb = path.stat().st_size / 1e6

    # Cold load
    evict(path)

    start = time.perf_counter()
    rb0 = io_field("read_bytes")

    load_file(path)

    cold = time.perf_counter() - start

    rb1 = io_field("read_bytes")

    cold_read = (
        (rb1 - rb0) / 1e6
        if rb0 is not None and rb1 is not None
        else None
    )

    # Warm load
    start = time.perf_counter()

    load_file(path)

    warm = time.perf_counter() - start

    # RSS + rchar
    gc.collect()

    rss_before = rss_mb()
    rchar_before = io_field("rchar")

    obj = load_file(path)

    rss_after = rss_mb()
    rchar_after = io_field("rchar")

    rss_delta = rss_after - rss_before

    rchar_delta = (
        (rchar_after - rchar_before) / 1e6
        if rchar_before is not None and rchar_after is not None
        else None
    )

    del obj
    gc.collect()

    # Explicit read() comparison
    rchar_before = io_field("rchar")

    _ = path.read_bytes()

    rchar_after = io_field("rchar")

    explicit_read_rchar = (
        (rchar_after - rchar_before) / 1e6
        if rchar_before is not None and rchar_after is not None
        else None
    )

    return {
        "file_mb": round(file_mb, 1),
        "cold_s": round(cold, 3),
        "warm_s": round(warm, 3),
        "cold_read_bytes_mb": round(cold_read, 1)
            if cold_read is not None else None,
        "load_rss_delta_mb": round(rss_delta, 1),
        "load_rchar_mb": round(rchar_delta, 1)
            if rchar_delta is not None else None,
        "explicit_read_rchar_mb": round(explicit_read_rchar, 1)
            if explicit_read_rchar is not None else None
    }

In [36]:
sd = model.state_dict()

RESULTS["formats"]["pt"] = measure_pt(
    sd,
    WORK / "model.pt"
)

RESULTS["formats"]["safetensors"] = measure_safetensors(
    sd,
    WORK / "model.safetensors"
)

for name, result in RESULTS["formats"].items():
    print(name)
    print(result)
    print()

pt
{'file_mb': 300.1, 'cold_s': 0.783, 'warm_s': 0.229, 'cold_read_bytes_mb': 300.1, 'load_rss_delta_mb': 286.1, 'load_rchar_mb': 300.1}

safetensors
{'file_mb': 300.1, 'cold_s': 0.003, 'warm_s': 0.0, 'cold_read_bytes_mb': 0.7, 'load_rss_delta_mb': 0.2, 'load_rchar_mb': 0.0, 'explicit_read_rchar_mb': 300.1}



In [37]:
big = torch.randn(8192, 8192)

v = big.T

RESULTS["stride"]["transpose_same_ptr"] = (
    v.data_ptr() == big.data_ptr()
)

sizes = []
times = []

for n in (2048, 4096, 6144, 8192):

    a = torch.randn(n, n)

    start = time.perf_counter()

    _ = a.T.contiguous()

    elapsed = (time.perf_counter() - start) * 1000

    bytes_moved = n * n * 4

    sizes.append(round(bytes_moved / 1e6, 1))
    times.append(round(elapsed, 2))

    del a
    gc.collect()


RESULTS["stride"]["contig_sizes_mb"] = sizes
RESULTS["stride"]["contig_times_ms"] = times


try:

    save_file(
        {"w": big.T},
        str(WORK / "probe.safetensors")
    )

    refused = False

except Exception as e:

    refused = True
    print("Expected rejection:")
    print(type(e).__name__)
    print(e)


save_file(
    {"w": big.contiguous()},
    str(WORK / "probe.safetensors")
)


RESULTS["stride"]["safetensors_refused_view"] = refused


print("Transpose shares buffer:",
      RESULTS["stride"]["transpose_same_ptr"])

print("Contiguous cost (MB -> ms):",
      list(zip(sizes, times)))

print("Safetensors refused view:",
      refused)

Expected rejection:
ValueError
You are trying to save a non contiguous tensor: `w` which is not allowed. It either means you are trying to save tensors which are reference of each other in which case it's recommended to save only the full tensors, and reslice at load time, or simply call `.contiguous()` on your tensor to pack it before saving.
Transpose shares buffer: True
Contiguous cost (MB -> ms): [(16.8, 52.04), (67.1, 196.04), (151.0, 439.9), (268.4, 417.21)]
Safetensors refused view: True


Prediction — Trust Boundary

I predict that the payload will not execute when torch.load uses
weights_only=True, because the weights-only loader restricts which
globals can be resolved.

I predict that the payload will execute when weights_only=False is
used because ordinary pickle deserialization can resolve and call
the callable returned by __reduce__.

In [38]:
PWNED = WORK / "PWNED.txt"


class Benign:

    def __reduce__(self):

        return (
            os.system,
            (f'echo "payload ran" > "{PWNED}"',)
        )


evil_path = WORK / "evil.pt"


torch.save(
    {
        "w": torch.zeros(2),
        "x": Benign()
    },
    evil_path
)


# Test 1: safe/default loading
PWNED.unlink(missing_ok=True)

try:

    torch.load(
        evil_path,
        weights_only=True
    )

    fired_default = PWNED.exists()
    blocked = False

except Exception:

    fired_default = PWNED.exists()
    blocked = True


# Test 2: explicitly unsafe loading
PWNED.unlink(missing_ok=True)

torch.load(
    evil_path,
    weights_only=False
)

fired_unsafe = PWNED.exists()


RESULTS["trust"] = {
    "payload_fired_default": bool(fired_default),
    "blocked_by_weights_only": bool(blocked),
    "payload_fired_unsafe": bool(fired_unsafe)
}


print(RESULTS["trust"])

if PWNED.exists():
    print("Payload created:", PWNED)
    print("Contents:", PWNED.read_text())

{'payload_fired_default': False, 'blocked_by_weights_only': True, 'payload_fired_unsafe': True}
Payload created: /content/lab11_12_work/PWNED.txt
Contents: payload ran



In [39]:
!wget -q --show-progress \
"https://huggingface.co/tensorblock/SmolLM-135M-GGUF/resolve/main/SmolLM-135M-Q2_K.gguf?download=true" \
-O /content/SmolLM-135M-Q2_K.gguf

/content/SmolLM-135 100%[===================>]  84.12M  55.8MB/s    in 1.5s    


In [40]:
GGUF_PATH = "/content/SmolLM-135M-Q2_K.gguf"

import os

print(
    "GGUF exists:",
    os.path.exists(GGUF_PATH)
)

if os.path.exists(GGUF_PATH):
    print(
        "GGUF size:",
        os.path.getsize(GGUF_PATH) / (1024**2),
        "MiB"
    )

GGUF exists: True
GGUF size: 84.11636352539062 MiB


In [41]:
def parse_gguf(path):

    UINT32 = 4
    FLOAT32 = 6
    STRING = 8
    ARRAY = 9

    f = open(path, "rb")

    assert f.read(4) == b"GGUF", "Not a GGUF file"

    version, = struct.unpack(
        "<I",
        f.read(4)
    )

    n_tensors, n_kv = struct.unpack(
        "<QQ",
        f.read(16)
    )

    def r_str():

        length, = struct.unpack(
            "<Q",
            f.read(8)
        )

        return f.read(length).decode(
            errors="replace"
        )

    unhandled = set()
    meta = {}

    def r_val(tag):

        if tag == UINT32:
            return struct.unpack(
                "<I",
                f.read(4)
            )[0]

        if tag == FLOAT32:
            return struct.unpack(
                "<f",
                f.read(4)
            )[0]

        if tag == STRING:
            return r_str()

        if tag == ARRAY:

            element_type, n = struct.unpack(
                "<IQ",
                f.read(12)
            )

            return [
                r_val(element_type)
                for _ in range(n)
            ]

        unhandled.add(tag)

        raise ValueError(tag)

    for _ in range(n_kv):

        key = r_str()

        tag, = struct.unpack(
            "<I",
            f.read(4)
        )

        try:

            meta[key] = r_val(tag)

        except ValueError:

            break

    f.close()

    architecture = meta.get(
        "general.architecture"
    )

    tokens = next(
        (
            value
            for key, value in meta.items()
            if key.endswith("tokens")
        ),
        []
    )

    return {
        "arch": architecture,
        "vocab_size": (
            len(tokens)
            if isinstance(tokens, list)
            else None
        ),
        "n_kv_read": len(meta),
        "unhandled_type_tags": sorted(unhandled)
    }

In [43]:
import struct

In [44]:
RESULTS["gguf_real"] = parse_gguf(GGUF_PATH)

print(RESULTS["gguf_real"])

{'arch': 'llama', 'vocab_size': 49153, 'n_kv_read': 22, 'unhandled_type_tags': [5]}


In [45]:
import os

print(os.path.exists(GGUF_PATH))
print(GGUF_PATH)

True
/content/SmolLM-135M-Q2_K.gguf


In [46]:
import os
import torch

PWNED = WORK / "PWNED.txt"

class Benign:
    def __reduce__(self):
        return (os.system, (f'echo "payload ran" > {PWNED}',))

# Create malicious-looking checkpoint
torch.save(
    {"w": torch.zeros(2), "x": Benign()},
    WORK / "evil.pt"
)

PWNED.unlink(missing_ok=True)

# Safe/default loading
try:
    torch.load(WORK / "evil.pt", weights_only=True)
    fired_default = PWNED.exists()
    blocked = False
except Exception:
    fired_default = PWNED.exists()
    blocked = True

PWNED.unlink(missing_ok=True)

# Explicitly unsafe loading
torch.load(WORK / "evil.pt", weights_only=False)
fired_unsafe = PWNED.exists()

RESULTS["trust"] = {
    "default_fired": fired_default,
    "blocked": blocked,
    "unsafe_fired": fired_unsafe,
}

print(RESULTS["trust"])

{'default_fired': False, 'blocked': True, 'unsafe_fired': True}


In [47]:
ANSWERS = {
    "warm_faster_than_cold": all(
        RESULTS["formats"][fmt]["warm_s"] <=
        RESULTS["formats"][fmt]["cold_s"] * 1.15
        for fmt in ["pt", "safetensors"]
    ),

    "safetensors_rss_near_zero": (
        RESULTS["formats"]["safetensors"]["load_rss_delta_mb"]
        < RESULTS["formats"]["safetensors"]["file_mb"] * 0.25
    ),

    "contiguous_cost_grows": (
        len(RESULTS["stride"]["contig_times_ms"]) >= 3
        and RESULTS["stride"]["contig_times_ms"][-1]
        > RESULTS["stride"]["contig_times_ms"][0]
    ),

    "weights_only_blocked_it": (
        RESULTS["trust"]["default_fired"] is False
        and RESULTS["trust"]["blocked"] is True
        and RESULTS["trust"]["unsafe_fired"] is True
    ),

    "safetensors_still_cannot": (
        "protect against malicious weights, surrounding repository code, "
        "or unknown provenance"
    ),
}

print(ANSWERS)

{'warm_faster_than_cold': True, 'safetensors_rss_near_zero': True, 'contiguous_cost_grows': True, 'weights_only_blocked_it': True, 'safetensors_still_cannot': 'protect against malicious weights, surrounding repository code, or unknown provenance'}


In [48]:
print("=== FORMATS ===")
print(RESULTS["formats"])

print("\n=== STRIDE ===")
print(RESULTS["stride"])

print("\n=== TRUST ===")
print(RESULTS["trust"])

print("\n=== GGUF ===")
print(RESULTS["gguf_real"])

print("\n=== ANSWERS ===")
print(ANSWERS)

=== FORMATS ===
{'pt': {'file_mb': 300.1, 'cold_s': 0.783, 'warm_s': 0.229, 'cold_read_bytes_mb': 300.1, 'load_rss_delta_mb': 286.1, 'load_rchar_mb': 300.1}, 'safetensors': {'file_mb': 300.1, 'cold_s': 0.003, 'warm_s': 0.0, 'cold_read_bytes_mb': 0.7, 'load_rss_delta_mb': 0.2, 'load_rchar_mb': 0.0, 'explicit_read_rchar_mb': 300.1}}

=== STRIDE ===
{'transpose_same_ptr': True, 'contig_sizes_mb': [16.8, 67.1, 151.0, 268.4], 'contig_times_ms': [52.04, 196.04, 439.9, 417.21], 'safetensors_refused_view': True}

=== TRUST ===
{'default_fired': False, 'blocked': True, 'unsafe_fired': True}

=== GGUF ===
{'arch': 'llama', 'vocab_size': 49153, 'n_kv_read': 22, 'unhandled_type_tags': [5]}

=== ANSWERS ===
{'warm_faster_than_cold': True, 'safetensors_rss_near_zero': True, 'contiguous_cost_grows': True, 'weights_only_blocked_it': True, 'safetensors_still_cannot': 'protect against malicious weights, surrounding repository code, or unknown provenance'}


In [52]:
import json
import platform
import sys
import safetensors

env = {
    "platform": platform.platform(),
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "safetensors": safetensors.__version__,
    "linux": IS_LINUX,
}

sub = {
    "roll": 202518044,
    "name": "NEEL",
    "env": env,
    "results": RESULTS,
    "answers": ANSWERS,
}

out = Path(f"submission_lab11_12_202518044.json")
out.write_text(json.dumps(sub, indent=2))

print("Submission created:")
print(out)

Submission created:
submission_lab11_12_202518044.json


In [53]:
import os

print("File:", out)
print("Exists:", os.path.exists(out))
print("Size:", os.path.getsize(out), "bytes")

File: submission_lab11_12_202518044.json
Exists: True
Size: 1523 bytes


In [54]:
from google.colab import files

files.download(str(out))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>